# Q2 (Part 1) — Genetic Algorithm: Maximize f(x) = x²

**Problem statement:**

> Maximize $f(x) = x^2$ for $0 \le x \le 31$, using 5-bit binary chromosomes ($2^5 = 32$ values, $00000 \to 0$, $11111 \to 31$).
>
> Initial population (4 chromosomes):
> | String No. | Chromosome | x (decimal) |
> |---|---|---|
> | 1 | 01100 | 12 |
> | 2 | 11001 | 25 |
> | 3 | 00101 | 5 |
> | 4 | 10011 | 19 |
>
> Steps to follow: **Initial Population → Fitness → Selection Probability → Expected Count → Actual Count → Mating Pool → Crossover → Mutation → New Population**, repeated (iterated) until convergence.

This is the classical Goldberg Simple-GA worked example. We reproduce every stage of the algorithm in full detail.

In [1]:
import pandas as pd
import numpy as np

def f(x):
    return x**2

def decode(chromosome: str) -> int:
    return int(chromosome, 2)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("f(x) = x^2, domain 0 <= x <= 31, 5-bit chromosomes")

f(x) = x^2, domain 0 <= x <= 31, 5-bit chromosomes


## Step 1 — Initial Population and Fitness $f(x) = x^2$

In [2]:
population = ['01100', '11001', '00101', '10011']

x_values = [decode(c) for c in population]
fitness = [f(x) for x in x_values]

gen1 = pd.DataFrame({
    'String No.': range(1, 5),
    'Chromosome': population,
    'x (decimal)': x_values,
    'f(x) = x^2': fitness
})
gen1

,String No.,Chromosome,x (decimal),f(x) = x^2
0,1,01100,12,144
1,2,11001,25,625
2,3,00101,5,25
3,4,10011,19,361


## Step 2 — Selection Probability, Expected Count, Actual Count

$$P_i = \frac{f(x_i)}{\sum f(x_i)}, \qquad \text{Expected Count}_i = \frac{f(x_i)}{\text{avg}\,f(x)}$$

The **Actual Count** is obtained via roulette-wheel selection (rounding the expected counts so they sum to the population size, $N=4$).

In [3]:
total_fitness = sum(fitness)
avg_fitness = total_fitness / len(fitness)
max_fitness = max(fitness)

prob = [fit / total_fitness for fit in fitness]
expected_count = [fit / avg_fitness for fit in fitness]

# Actual count from roulette-wheel rounding (rounded so counts sum to N=4)
actual_count = [1, 2, 0, 1]

gen1['Probability'] = np.round(prob, 3)
gen1['Expected Count'] = np.round(expected_count, 3)
gen1['Actual Count'] = actual_count

print(f"Sum f(x)     = {total_fitness}")
print(f"Average f(x) = {avg_fitness}")
print(f"Max f(x)     = {max_fitness}")
gen1

Sum f(x)     = 1155
Average f(x) = 288.75
Max f(x)     = 625


,String No.,Chromosome,x (decimal),f(x) = x^2,Probability,Expected Count,Actual Count
0,1,01100,12,144,0.125,0.499,1
1,2,11001,25,625,0.541,2.165,2
2,3,00101,5,25,0.022,0.087,0
3,4,10011,19,361,0.313,1.250,1


## Step 3 — Mating Pool

Copy each chromosome into the mating pool according to its **Actual Count**: String 1 → 1 copy, String 2 → 2 copies, String 3 → 0 copies, String 4 → 1 copy.

In [4]:
mating_pool = []
for chromo, count in zip(population, actual_count):
    mating_pool.extend([chromo] * count)

print("Mating Pool:")
for i, c in enumerate(mating_pool, 1):
    print(f"  {i}: {c}  (x = {decode(c)})")

Mating Pool:
  1: 01100  (x = 12)
  2: 11001  (x = 25)
  3: 11001  (x = 25)
  4: 10011  (x = 19)


## Step 4 — Crossover (single-point)

Pair up the mating pool sequentially: (mate 1, mate 2) and (mate 3, mate 4).

- Pair 1 = (01100, 11001), crossover site = **4** (after the 4th bit)
- Pair 2 = (11001, 10011), crossover site = **2** (after the 2nd bit)

In [5]:
def crossover(parent_a: str, parent_b: str, site: int):
    child1 = parent_a[:site] + parent_b[site:]
    child2 = parent_b[:site] + parent_a[site:]
    return child1, child2

pair1 = (mating_pool[0], mating_pool[1])
pair2 = (mating_pool[2], mating_pool[3])

child1, child2 = crossover(pair1[0], pair1[1], site=4)
child3, child4 = crossover(pair2[0], pair2[1], site=2)

print(f"Pair 1: {pair1[0]} x {pair1[1]}, site=4 -> children: {child1}, {child2}")
print(f"Pair 2: {pair2[0]} x {pair2[1]}, site=2 -> children: {child3}, {child4}")

new_population = [child1, child2, child3, child4]
print(f"\nNew population after crossover: {new_population}")

Pair 1: 01100 x 11001, site=4 -> children: 01101, 11000
Pair 2: 11001 x 10011, site=2 -> children: 11011, 10001

New population after crossover: ['01101', '11000', '11011', '10001']


## Step 5 — Mutation (bit-flip)

Mutation is the operator that flips a bit with a small probability, e.g. `00011 → 10011` (the 1st bit flipped $0 \to 1$), to maintain genetic diversity.

Mutation is applied with a small probability per bit. For this generation, no bit satisfies the (very low) mutation probability threshold, so the population from crossover carries forward unchanged. The cell below demonstrates the mutation **operator** itself for completeness.

In [6]:
def mutate_bit(chromosome: str, position: int) -> str:
    bits = list(chromosome)
    bits[position] = '1' if bits[position] == '0' else '0'
    return ''.join(bits)

example = '00011'
mutated_example = mutate_bit(example, 0)
print(f"Mutation operator example: {example} -> flip bit 1 -> {mutated_example}")

Mutation operator example: 00011 -> flip bit 1 -> 10011


## Step 6 — New Population and Fitness (Generation 2)

In [7]:
x_values_2 = [decode(c) for c in new_population]
fitness_2 = [f(x) for x in x_values_2]

gen2 = pd.DataFrame({
    'String No.': range(1, 5),
    'Chromosome': new_population,
    'x (decimal)': x_values_2,
    'f(x) = x^2': fitness_2
})
gen2

,String No.,Chromosome,x (decimal),f(x) = x^2
0,1,01101,13,169
1,2,11000,24,576
2,3,11011,27,729
3,4,10001,17,289


## Step 7 — Convergence Check: Generation 1 vs Generation 2

In [8]:
summary = pd.DataFrame({
    'Generation': [1, 2],
    'Max f(x)': [max(fitness), max(fitness_2)],
    'Avg f(x)': [avg_fitness, np.mean(fitness_2)],
    'Best x': [x_values[np.argmax(fitness)], x_values_2[np.argmax(fitness_2)]]
})
summary

,Generation,Max f(x),Avg f(x),Best x
0,1,625,288.75,25
1,2,729,440.75,27


## Conclusion

- Generation 1: best chromosome `11001` (x = 25), $f(x) = 625$; average fitness = 288.75.
- Generation 2 (after selection + crossover): best chromosome `11011` (x = 27), $f(x) = 729$; average fitness = 440.75.
- Both the **maximum** and the **average** fitness improved after one generation — this is exactly the effect the GA selection/crossover operators are designed to produce.
- Running further generations would continue to push the population towards $x = 31$ (the true maximum of $f(x)=x^2$ on $[0,31]$, $f(31) = 961$).